# 연습문제 2-4 풀이: 가중치 초기화와 시그모이드 함수의 관계

2-2절 연습문제 2-4의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch02/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트
import torch

## 연습문제 2-4

> `torch.randn()` 함수로 만든 텐서의 요솟값은 평균 0, 표준편차 1인 정규분포를 따르며,
> 각 요소의 값이 -2~2 사이일 확률은 95%이다. 초깃값이 평균 0, 표준편차 10인 정규분포를 따르도록
> 가중치 초기화를 바꾸면 학습 결과가 어떻게 될까? 초기화 방법을 바꿔 서로 다른 초깃값으로
> 10번 이상 반복 학습해 결과를 확인해 보자.
>
> 힌트: 표준편차가 10인 값으로 초기화한 텐서가 파라미터로 사용하기 적절한지 확인해 볼 필요가 있다.

## 사전 준비

2-2절의 AND 게이트 데이터, 모델 함수, 손실 함수, 최적화 함수, 학습 루프를 그대로 옮겨 온다.
학습 로그는 시도마다 수백 줄이 되므로, 마지막 손실만 돌려주는 조용한 학습 함수를 따로 구현해 사용한다. 학습한 모델이 네 샘플을 모두 바르게 분류하면 모델 구현 성공으로 판단한다.

In [2]:
LR = 0.1          # 학습률
EPOCHS = 1000     # 전체 학습 에포크

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])    # (4, 2): 입력
Y_true = torch.tensor([[0.], [0.], [0.], [1.]])               # (4, 1): AND 게이트 정답

def model(X, weights, bias):
    return 1 / (1 + torch.exp(-(X @ weights + bias)))         # 가중합과 시그모이드

def criterion(Y_pred, Y_true):
    return ((Y_pred - Y_true) ** 2).mean()                    # 평균제곱오차

def optimizer(weights, bias, learning_rate):
    with torch.no_grad():
        weights -= learning_rate * weights.grad
        bias -= learning_rate * bias.grad

In [3]:
SEED = 2          # 초깃값을 재현할 수 있도록 고정하는 시드

def train_quiet(weights, bias, epochs=EPOCHS):
    """[코드 2-5]와 같은 학습 루프. 로그 대신 마지막 손실만 반환한다."""
    for _ in range(epochs):
        loss = criterion(model(X, weights, bias), Y_true)
        loss.backward()
        optimizer(weights, bias, LR)
        weights.grad.zero_()
        bias.grad.zero_()
    return loss.item()

def is_correct(weights, bias):
    """0.5를 문턱값으로 네 샘플을 모두 바르게 분류하는지 확인한다."""
    with torch.no_grad():
        return torch.equal((model(X, weights, bias) >= 0.5).float(), Y_true)

**표준편차 10으로 초기화해 10번 학습하기**
- `torch.randn()`의 결과에 10을 곱하면 평균 0, 표준편차 10인 초깃값이 된다
- 초깃값(`w1`, `w2`, `b`)을 매번 새로 뽑아 같은 조건(학습률 0.1, 1,000 에포크)으로 학습
- 성공 기준: 0.5를 문턱값으로 네 샘플을 모두 바르게 분류

In [4]:
INIT_STD = 10.              # 초깃값의 표준편차
TRIALS = 10                 # 반복 횟수

torch.manual_seed(SEED)
results = []                # 시도마다 (w1, w2, b, 학습 후 손실, 분류 성공 여부)
for _ in range(TRIALS):
    # torch.randn()에 표준편차를 곱하면 평균 0, 표준편차 INIT_STD인 난수가 된다
    weights = (torch.randn((2, 1)) * INIT_STD).requires_grad_(True)
    bias = (torch.randn((1,)) * INIT_STD).requires_grad_(True)
    w1, w2, b = weights[0].item(), weights[1].item(), bias.item()
    loss = train_quiet(weights, bias)
    results.append((w1, w2, b, loss, is_correct(weights, bias)))

success = sum(r[4] for r in results)
print(f'{TRIALS}번 중 분류에 성공한 횟수: {success}번')

10번 중 분류에 성공한 횟수: 2번


**시도별 결과 보기**
- 시도마다 초깃값, 학습 후 손실, 분류 성공 여부를 표로 출력

In [5]:
print('시도   학습 후 손실   분류 성공     초깃값 (w1, w2, b)')
for i, (w1, w2, b, loss, ok) in enumerate(results, 1):
    print(f'{i:4d}   {loss:12.4f}   {str(ok):>9}     ({w1:6.1f}, {w2:6.1f}, {b:6.1f})')

시도   학습 후 손실   분류 성공     초깃값 (w1, w2, b)
   1         0.0358        True     (   3.9,   -2.2,   -3.2)
   2         0.2590       False     ( -12.1,   10.4,   -6.3)
   3         0.0074        True     (   5.7,    5.4,   -3.9)
   4         0.5001       False     ( -10.4,   13.2,    7.5)
   5         0.2529       False     ( -13.3,  -12.4,   -1.0)
   6         0.2547       False     (  -9.5,    6.2,   -2.4)
   7         0.2527       False     (   0.2,   -4.9,   -0.6)
   8         0.2500       False     ( -15.7,    4.3,  -14.8)
   9         0.2512       False     (  -4.8,    2.5,    1.6)
  10         0.7499       False     (  -1.6,    4.2,   10.0)


**결과 분석: 열 번 중 두 번만 성공한다**
- 10번 중 성공은 2번(시도 1, 3), 나머지 8번은 학습에 실패
- 실패한 경우의 손실은 0.25, 0.50, 0.75 근처: 네 샘플 중 1, 2, 3개가 틀린 상태
- 출력이 0이나 1에 붙어 버려 1,000 에포크가 지나도 손실이 그 값에 머뭄

**원인 찾기: 초깃값에서 시그모이드가 평평한 샘플**
- 힌트대로 표준편차 10인 초깃값이 파라미터로 적절한지 확인
- 표준편차 1과 10으로 초깃값을 10번씩 뽑아, 시그모이드 기울기가 0.001보다 작은(평평한) 샘플을 센다
- 처음부터 틀렸는데 평평한 샘플이 있는 시도를 '갇힌 시도'로 센다: 그 샘플은 학습으로 바로잡히지 않는다

In [6]:
def check_init(std, trials=10):
    """초깃값에서 시그모이드가 평평한(기울기 0.001 미만) 샘플을 센다."""
    torch.manual_seed(SEED)                       # run_trials()와 같은 초깃값
    flat, stuck = 0, 0
    for _ in range(trials):
        weights = torch.randn((2, 1)) * std
        bias = torch.randn((1,)) * std
        Y_pred = model(X, weights, bias)
        is_flat = Y_pred * (1 - Y_pred) < 1e-3    # 시그모이드 기울기가 거의 0인 샘플
        is_wrong = (Y_pred >= 0.5).float() != Y_true   # 처음부터 틀린 샘플
        flat += is_flat.sum().item()
        stuck += (is_flat & is_wrong).any().item()     # 갇힌 시도
    print(f'표준편차 {std:2.0f}: 평평한 샘플 {flat}개, 갇힌 시도 {stuck}번')

check_init(1.)
check_init(10.)

표준편차  1: 평평한 샘플 0개, 갇힌 시도 0번
표준편차 10: 평평한 샘플 18개, 갇힌 시도 5번


**시그모이드의 기울기**
- 시그모이드 y의 기울기는 y * (1 - y)
- z가 0에서 멀어질 때 기울기가 얼마나 줄어드는지 확인

In [7]:
# 아주 작은 기울기까지 보려고 float64를 쓴다
z = torch.tensor([0., 2., 5., 10., 20.], dtype=torch.float64)
y = torch.sigmoid(z)
for zi, yi in zip(z.tolist(), y.tolist()):
    print(f'z = {zi:4.0f}: y = {yi:.8f}, 기울기 y(1 - y) = {yi * (1 - yi):.2e}')

z =    0: y = 0.50000000, 기울기 y(1 - y) = 2.50e-01
z =    2: y = 0.88079708, 기울기 y(1 - y) = 1.05e-01
z =    5: y = 0.99330715, 기울기 y(1 - y) = 6.65e-03
z =   10: y = 0.99995460, 기울기 y(1 - y) = 4.54e-05
z =   20: y = 1.00000000, 기울기 y(1 - y) = 2.06e-09


**결과 분석: 가중합이 크면 기울기가 거의 0이다**
- z = 0에서 기울기는 0.25로 가장 크고, z = 10이면 4.5e-05, z = 20이면 2.1e-09
- 시그모이드 함수는 입력이 양수든, 음수든 0에서 멀어지면 기울기가 0에 가까워지는 함수([그림 2-4] 참조) --> 기울기가 0에 가까워져 매 에포크마다 파라미터의 값도 거의 변하지 않음(평평한 곳, 갇힌 시도)
- 표준편차 10인 초깃값은 40개 샘플(10번 x 4개) 중 18개가 평평한 곳에서 시작(표준편차 1은 0개)
- 갇힌 시도가 10번 중 5번(표준편차 1은 0번): 기울기가 거의 0이라 틀린 샘플이 바로잡히지 않음
- 나머지 실패는 틀린 샘플의 기울기가 매우 작은 경우이거나 기울기는 있지만 학습이 느린 경우

**표준편차를 바꿔 가며 비교하기**
- 같은 난수열에 표준편차만 1, 3, 5, 10으로 바꿔 10번씩 학습
- 표준편차가 커질수록 성공 횟수가 어떻게 달라지는지 확인

In [8]:
def run_trials(std, epochs=EPOCHS, trials=10):
    """표준편차 std로 초기화해 trials번 학습하고 (성공 횟수, 손실 목록)을 돌려준다."""
    torch.manual_seed(SEED)                  # 표준편차만 다르고 난수열은 같다
    success, losses = 0, []
    for _ in range(trials):
        weights = (torch.randn((2, 1)) * std).requires_grad_(True)
        bias = (torch.randn((1,)) * std).requires_grad_(True)
        losses.append(train_quiet(weights, bias, epochs))
        success += is_correct(weights, bias)
    return success, losses

for std in (1., 3., 5., 10.):
    success, _ = run_trials(std)
    print(f'표준편차 {std:4.0f}: 10번 중 {success}번 성공')

표준편차    1: 10번 중 10번 성공
표준편차    3: 10번 중 6번 성공
표준편차    5: 10번 중 3번 성공
표준편차   10: 10번 중 2번 성공


**결과 확인: 표준편차가 클수록 모델 학습이 잘 되지 않는다**
- 표준편차 1은 10번 모두 성공, 3은 6번, 5는 3번, 10은 2번
- 표준편차가 클수록 초깃값도 커질 확률이 높아짐
- 초깃값이 클수록 시그모이드가 포화된 곳에서 시작하는 경우가 늘어남
- `torch.randn()`의 기본 표준편차 1이 이 모델의 파라미터에 적절한 크기

**직접 해 보기: 에포크를 10배로 늘리면?**
- 표준편차 10, 학습률 0.1은 그대로 두고 에포크만 1,000에서 10,000으로 변경
- 성공 횟수와 시도별 손실이 어떻게 될까?

In [9]:
for epochs in (EPOCHS, EPOCHS * 10):
    success, losses = run_trials(10., epochs)
    loss_text = ' '.join(f'{loss:.2f}' for loss in losses)
    print(f'에포크 {epochs:6,}: {success}번 성공, 손실 {loss_text}')

에포크  1,000: 2번 성공, 손실 0.04 0.26 0.01 0.50 0.25 0.25 0.25 0.25 0.25 0.75
에포크 10,000: 3번 성공, 손실 0.01 0.25 0.00 0.50 0.25 0.25 0.25 0.25 0.01 0.75


**확인**
- 에포크를 10배로 늘려도 성공은 2번에서 3번으로 딱 한 경우만 추가
- 나머지 실패는 여전히 손실 0.25, 0.50, 0.75 근처에 머뭄
- 기울기가 거의 0인 곳에서는 에포크를 늘려도 학습되지 않음: 시간이 더 걸리는 정도가 아니라 수렴하지 못함